[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asthakalita22/deepfake-detection/blob/main/deepfake_detection.ipynb)

# Deepfake Detection (FaceForensics++ C23)

Pipeline: face-cropped individual frames -> leakage-free split by identity -> pretrained EfficientNetB0 (2-phase fine-tuning) -> threshold chosen on validation -> frame-level and video-level evaluation on test.

**Setup:** add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab (left sidebar, key icon -> Secrets) and enable notebook access. Never paste `kaggle.json` into a cell.

In [ ]:
!pip install -q kagglehub opencv-python scikit-learn tqdm

In [ ]:
import os, glob, re, json, random
import cv2, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 224
QUICK = True   # True: ~25-30 min total on a Colab GPU. False: bigger dataset, better accuracy, slower.
FRAMES_PER_VIDEO = 6 if QUICK else 10
BATCH = 32
FRAMES_DIR = "/content/frames"
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Download dataset (credentials from Colab secrets)

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
path = kagglehub.dataset_download("xdxd003/ff-c23")
data_dir = os.path.join(path, "FaceForensics++_C23")
print(data_dir, os.listdir(data_dir))

## 2. Build video list and a leakage-free split

FF++ fakes are named `<target>_<source>.mp4` and both ids refer to original videos; DeepFakeDetection clips are named `<actorA>_<actorB>__scene__id.mp4`. 
Each identity (video id / actor) is assigned to exactly one split, and a video is kept only if **all** of its identities fall in the same split. 
This stops the same face/source from appearing in both train and test.

In [ ]:
FAKE_CATS = ["DeepFakeDetection", "Deepfakes", "Face2Face", "FaceShifter", "FaceSwap", "NeuralTextures"]
MAX_PER_FAKE_CAT = 80 if QUICK else 250   # FF++ has ~1000 per category
MAX_REAL = 400 if QUICK else 1000

def identities(category, stem):
    if category == "DeepFakeDetection":
        return {"dfd_" + a for a in stem.split("__")[0].split("_")}
    return {"ff_" + t for t in re.findall(r"\d+", stem)}

rows = []
for cat in ["original"] + FAKE_CATS:
    vids = sorted(glob.glob(os.path.join(data_dir, cat, "*.mp4")))
    random.Random(SEED).shuffle(vids)
    vids = vids[:MAX_REAL] if cat == "original" else vids[:MAX_PER_FAKE_CAT]
    for v in vids:
        stem = os.path.splitext(os.path.basename(v))[0]
        rows.append(dict(video=v, category=cat, label=0 if cat == "original" else 1,
                         ids=identities(cat, stem)))
videos = pd.DataFrame(rows)

# assign every identity to a split (70/15/15)
all_ids = sorted(set().union(*videos["ids"]))
rng = random.Random(SEED); rng.shuffle(all_ids)
n = len(all_ids)
id_split = {i: ("train" if k < 0.70 * n else "val" if k < 0.85 * n else "test")
            for k, i in enumerate(all_ids)}

def video_split(ids):
    s = {id_split[i] for i in ids}
    return s.pop() if len(s) == 1 else None   # None -> straddles splits, drop

videos["split"] = videos["ids"].apply(video_split)
dropped = videos["split"].isna().sum()
videos = videos.dropna(subset=["split"]).drop(columns="ids").reset_index(drop=True)
print(f"dropped {dropped} videos that straddled splits")
print(videos.groupby(["split", "label"]).size().unstack())

## 3. Extract face-cropped frames

Individual frames (not averaged), face detected with OpenCV's Haar cascade, box expanded by 30%, falls back to a centre crop if no face is found. Filenames include the category so nothing gets overwritten.

In [ ]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def crop_face(frame, margin=0.3):
    h, w = frame.shape[:2]
    scale = 480 / max(h, w)
    small = cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    if len(faces) == 0:
        s = min(h, w); y0, x0 = (h - s) // 2, (w - s) // 2
        return frame[y0:y0 + s, x0:x0 + s], False
    x, y, fw, fh = max(faces, key=lambda f: f[2] * f[3])
    x, y, fw, fh = [int(v / min(scale, 1)) for v in (x, y, fw, fh)]
    cx, cy, side = x + fw // 2, y + fh // 2, int(max(fw, fh) * (1 + margin))
    x0, y0 = max(cx - side // 2, 0), max(cy - side // 2, 0)
    return frame[y0:y0 + side, x0:x0 + side], True

def extract(row):
    out = []
    stem = os.path.splitext(os.path.basename(row.video))[0]
    cap = cv2.VideoCapture(row.video)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < FRAMES_PER_VIDEO:
        cap.release(); return out
    targets = set(np.linspace(0, total - 1, FRAMES_PER_VIDEO).astype(int).tolist())
    last = max(targets)
    for i in range(last + 1):
        if not cap.grab():
            break
        if i in targets:
            ok, frame = cap.retrieve()
            if not ok:
                continue
            face, found = crop_face(frame)
            if face.size == 0:
                continue
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
            d = os.path.join(FRAMES_DIR, row.split, str(row.label))
            os.makedirs(d, exist_ok=True)
            p = os.path.join(d, f"{row.category}_{stem}_{i:05d}.jpg")
            cv2.imwrite(p, face, [cv2.IMWRITE_JPEG_QUALITY, 95])
            out.append(dict(path=p, label=row.label, split=row.split, video=row.video,
                            category=row.category, face_found=found))
    cap.release()
    return out

meta_csv = os.path.join(FRAMES_DIR, "frames.csv")
if os.path.exists(meta_csv):
    frames = pd.read_csv(meta_csv)
else:
    os.makedirs(FRAMES_DIR, exist_ok=True)
    with ThreadPoolExecutor(max_workers=8) as ex:
        results = list(tqdm(ex.map(extract, videos.itertuples()), total=len(videos)))
    frames = pd.DataFrame([r for rs in results for r in rs])
    frames.to_csv(meta_csv, index=False)

print(frames.groupby(["split", "label"]).size().unstack())
print("face detected in %.1f%% of frames" % (100 * frames["face_found"].mean()))

In [ ]:
# sanity check: a few training crops
import matplotlib.pyplot as plt
sample = frames[frames.split == "train"].groupby("label").sample(4, random_state=SEED)
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, r) in zip(axs.ravel(), sample.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{'fake' if r.label else 'real'} / {r.category}", fontsize=9); ax.axis("off")
plt.show()

## 4. Input pipeline

In [ ]:
def make_ds(df, train):
    paths = df["path"].values
    labels = df["label"].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    def load(p, y):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        return img, y            # EfficientNet expects raw 0-255 pixels
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    return ds.prefetch(tf.data.AUTOTUNE)

train_df = frames[frames.split == "train"].reset_index(drop=True)
val_df   = frames[frames.split == "val"].reset_index(drop=True)
test_df  = frames[frames.split == "test"].reset_index(drop=True)
train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

counts = np.bincount(train_df["label"])
class_weight = {0: len(train_df) / (2 * counts[0]), 1: len(train_df) / (2 * counts[1])}
print("class_weight:", class_weight)

## 5. Model: pretrained EfficientNetB0

Phase 1 trains only the new head with the backbone frozen. Phase 2 unfreezes the top of the backbone (BatchNorm layers stay frozen) and fine-tunes at a low learning rate.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augment")

base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

METRICS = [keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc")]
CKPT = "/content/best.weights.h5"

def callbacks():
    return [
        keras.callbacks.ModelCheckpoint(CKPT, monitor="val_auc", mode="max",
                                        save_best_only=True, save_weights_only=True),
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
    ]

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=4 if QUICK else 8,
               class_weight=class_weight, callbacks=callbacks())

In [ ]:
# Phase 2: fine-tune the top of the backbone
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=METRICS)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=8 if QUICK else 20,
               class_weight=class_weight, callbacks=callbacks())

In [ ]:
# training curves
hist = {k: h1.history[k] + h2.history[k] for k in h1.history if k in h2.history}
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axs, ["loss", "acc", "auc"]):
    ax.plot(hist[k], label="train"); ax.plot(hist["val_" + k], label="val")
    ax.axvline(len(h1.history["loss"]) - 0.5, ls="--", c="gray"); ax.set_title(k); ax.legend()
plt.show()

## 6. Evaluation

The decision threshold is chosen on the **validation** set (max F1) and then applied unchanged to the test set. Video-level scores average the frame probabilities of each video.

In [ ]:
val_prob  = model.predict(val_ds).ravel()
test_prob = model.predict(test_ds).ravel()

ts = np.linspace(0.05, 0.95, 91)
best_t = ts[np.argmax([f1_score(val_df["label"], val_prob > t) for t in ts])]
print(f"threshold chosen on val: {best_t:.2f}")

def report(y, p, t, title):
    pred = (p > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    print(f"--- {title} ---")
    print(f"Acc {accuracy_score(y, pred):.4f} | Prec {precision_score(y, pred):.4f} | "
          f"Recall {recall_score(y, pred):.4f} | Spec {tn / (tn + fp):.4f} | "
          f"F1 {f1_score(y, pred):.4f} | AUC {roc_auc_score(y, p):.4f}")
    print(f"Confusion [[TN FP][FN TP]]: [[{tn}, {fp}], [{fn}, {tp}]]")

report(test_df["label"].values, test_prob, best_t, "Test, frame level")

test_df["prob"] = test_prob
vid = test_df.groupby("video").agg(prob=("prob", "mean"), label=("label", "first"),
                                   category=("category", "first")).reset_index()
val_vid = val_df.assign(prob=val_prob).groupby("video").agg(prob=("prob", "mean"), label=("label", "first"))
vt = ts[np.argmax([f1_score(val_vid["label"], val_vid["prob"] > t) for t in ts])]
report(vid["label"].values, vid["prob"].values, vt, f"Test, video level (threshold {vt:.2f})")

print("\nVideo-level recall per fake method:")
vid["pred"] = (vid["prob"] > vt).astype(int)
print(vid[vid.label == 1].groupby("category")["pred"].mean().round(3))

In [ ]:
fpr, tpr, _ = roc_curve(test_df["label"], test_prob)
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(test_df['label'], test_prob):.3f}")
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.title("ROC (test, frame level)")
plt.show()

model.save("/content/deepfake_effnetb0.keras")